# Day 41：LangGraph 入门（重点）（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 6 周**：Agent 基础、工具调用与 LangGraph 状态机  
> **GitLab 工单**：`STARMIND-041` ｜ **开发分支**：`feat/day41-langgraph-state-machine`  
> **配套 HTML5 课件**：`courseware/day41/index.html` ｜ **独立版本快照**：`versions/day41/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天打起十二分精神，因为 LangGraph 是整个第四阶段、乃至当前国内大厂落地生产级 Agent 的头号核心技术。很多同学一听“图编排”觉得玄乎，其实它就三个概念：`State`（全局共享账本）、`Node`（干活的纯函数）、`Edge`（决定下一站去哪的路由器）。今天老陈带大家亲手把 LangGraph 的 `StateGraph` 内核写出来，再分别用它重写 ReAct 循环和搭建带人工审批节点的采购工作流，最后还能一键导出 Mermaid 和 ASCII 架构图！

### 承上启下
- **昨日基线**：昨天（Day 40）我们深入实现了 `@tool` 装饰器和 `AgentExecutor`，但在结尾我们发现黑盒循环在面对“确定性业务规则 + 自由推理 + 人工审批分支”的混合场景时根本无法精细控场。
- **明日预告**：明天（Day 42）我们进入 LangGraph 进阶：给状态图加上 SQLite Checkpointer 断点持久化、真正的 `interrupt_before` 异步人工审批恢复、Subgraph 子图封装，以及 Fan-out/Fan-in 并行写长文 Agent。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day41'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-041）

星辰数科 IT 采购与运维系统准备接入星辰智脑：员工提出设备采购或资料查询需求时，系统既要具备 ReAct 自主查库存/查价格的能力，又要对超过 5000 元的采购单强制路由到总监人工审批节点，严禁大模型绕过审批直接下单。

**今日迭代目标：**
- 理解有向循环图（Cyclic State Machine）相比 DAG 流水线的架构优势。
- 实现 100% 对齐 LangGraph 核心语义的 `StateGraph` 引擎（`START`/`END`、`add_node`、`add_edge`、`add_conditional_edges`、Reducer 状态合并、`draw_mermaid`、`draw_ascii`）。
- 使用状态图分别实现：1）循环版 ReAct Agent；2）带大小额条件路由与人工审批节点的企业采购工作流。

**验收标准：**
- `StateGraph` 支持条件边动态路由与带最大步数保护的循环边，且能同时导出合法的 Mermaid 语法与清晰的 ASCII 终端流程图。
- ReAct 状态图能完成 `agent -> tools -> agent -> END` 的闭环推理。
- 采购审批状态图对金额 <= 5000 元走 `auto_approve`，金额 > 5000 元强制路由到 `human_approval` 节点并按审批结果流向 `execute_order` 或 `reject_notify`。

---

## 📚 课堂核心技术笔记

### 1. 1. 真实 LangGraph 生产 API 对照（langgraph >= 0.2）

真实 `langgraph` 库的代码风格与我们今天手写的引擎几乎一字不差：
```python
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    next_action: str

builder = StateGraph(AgentState)
builder.add_node('agent', agent_node)
builder.add_node('tools', tool_node)
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', route_fn, {'tools': 'tools', 'end': END})
builder.add_edge('tools', 'agent')
app = builder.compile()
print(app.get_graph().draw_mermaid())
```

> **⚠️ 老陈避坑**：坑点：在真实 LangGraph 中，如果不给 `messages` 加 `Annotated[list, add_messages]` 或 `operator.add`，节点返回 `{'messages': [new_msg]}` 会直接把之前的对话历史全清空覆盖！

### 2. 2. 条件边必须显式提供 path_map（路径映射表）

虽然某些框架允许省略 `path_map` 直接让路由函数返回节点名字符串，但老陈强烈要求团队在 `add_conditional_edges` 里显式写出 `path_map={'continue': 'tools', 'finish': END}`。原因有两个：第一，编译期可以做静态合法性检查；第二，`draw_mermaid()` 只有看到 `path_map` 才能把所有分支虚线准确画出来。

> **⚠️ 老陈避坑**：坑点：如果路由函数返回了一个不在 `path_map` 里的键值，引擎应该立刻抛出清晰的 `ValueError`，而不是静默跳过导致流程假死。

### 3. 3. 递归上限 recursion_limit 是状态循环图的生命线

LangGraph 支持环路（Cycle），而有环就一定存在死循环风险。真实 LangGraph 默认 `recursion_limit=25`（通过 `app.invoke(state, config={'recursion_limit': 10})` 配置）。在我们的引擎中同样内置 `max_steps` 保护，超限即抛 `RecursionError` 或安全中止。

> **⚠️ 老陈避坑**：坑点：在测试阶段故意把 `max_steps` 设小（例如 6~10），一旦 Prompt 有 bug 立刻就能在单测里炸出来，绝不要上线前设成 100。


## 💻 实操步骤 1：步骤一：实现 StateGraph 状态图核心引擎与双格式可视化

- **目标文件**：`versions/day41/src/day41_state_graph.py`
- **老陈代码旁白**：先来造一颗麻雀虽小五脏俱全的 `StateGraph` 引擎！它支持 `START`/`END` 哨兵节点、Reducer 列表追加合并、普通边、条件边、编译期拓扑校验，以及一键导出 `draw_mermaid()` 和 `draw_ascii()`。


In [ ]:
"""Day 41 - 步骤一：轻量级 LangGraph 语义 StateGraph 引擎与 Mermaid/ASCII 可视化。"""

from __future__ import annotations

import copy
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Set, Tuple

START = "__start__"
END = "__end__"

NodeFunc = Callable[[Dict[str, Any]], Dict[str, Any]]
RouterFunc = Callable[[Dict[str, Any]], str]
ReducerFunc = Callable[[Any, Any], Any]


@dataclass
class CompiledGraph:
    """编译后的可执行状态图对象。"""

    nodes: Dict[str, NodeFunc]
    edges: Dict[str, str]
    conditional_edges: Dict[str, Tuple[RouterFunc, Dict[str, str]]]
    reducers: Dict[str, ReducerFunc]

    def _merge_state(self, state: Dict[str, Any], update: Dict[str, Any]) -> Dict[str, Any]:
        """按字段 Reducer 合并节点返回的局部更新补丁。"""
        merged = copy.deepcopy(state)
        for key, value in update.items():
            if key in self.reducers and key in merged:
                merged[key] = self.reducers[key](merged[key], value)
            else:
                merged[key] = value
        return merged

    def invoke(self, initial_state: Dict[str, Any], max_steps: int = 15) -> Dict[str, Any]:
        """从 START 节点启动图执行，直至抵达 END 或触发最大步数保护。"""
        state = copy.deepcopy(initial_state)
        state.setdefault("_visited_nodes", [])
        current = self._next_node(START, state)
        steps = 0

        while current != END:
            steps += 1
            if steps > max_steps:
                raise RuntimeError(f"状态图超过最大步数上限 ({max_steps})，疑似陷入循环")

            node_fn = self.nodes[current]
            update = node_fn(copy.deepcopy(state))
            if not isinstance(update, dict):
                raise TypeError(f"节点 '{current}' 必须返回 dict 状态补丁，实际返回 {type(update).__name__}")
            state = self._merge_state(state, update)
            state["_visited_nodes"].append(current)
            current = self._next_node(current, state)

        return state

    def _next_node(self, current: str, state: Dict[str, Any]) -> str:
        if current in self.conditional_edges:
            router_fn, path_map = self.conditional_edges[current]
            route_key = router_fn(state)
            if route_key not in path_map:
                raise ValueError(f"节点 '{current}' 路由返回未知分支 '{route_key}'，合法值: {list(path_map.keys())}")
            return path_map[route_key]
        if current in self.edges:
            return self.edges[current]
        raise ValueError(f"节点 '{current}' 未配置任何出边，无法继续流转")

    def draw_mermaid(self) -> str:
        """导出标准 Mermaid flowchart TD 文本，可直接贴入文档或前端渲染。"""
        lines = ["graph TD"]
        for src, dst in self.edges.items():
            lines.append(f"    {src} --> {dst}")
        for src, (_, path_map) in self.conditional_edges.items():
            for cond_label, dst in path_map.items():
                lines.append(f"    {src} -. {cond_label} .-> {dst}")
        return "\n".join(lines)

    def draw_ascii(self) -> str:
        """导出简洁的 ASCII 拓扑结构图，便于终端调试与日志查看。"""
        lines = ["+--- StateGraph Topology ---+"]
        for src, dst in self.edges.items():
            lines.append(f"| [{src}] ===> [{dst}]")
        for src, (_, path_map) in self.conditional_edges.items():
            for cond_label, dst in path_map.items():
                lines.append(f"| [{src}] --({cond_label})--> [{dst}]")
        lines.append("+---------------------------+")
        return "\n".join(lines)


class StateGraph:
    """对齐 LangGraph StateGraph 的构建器 API。"""

    def __init__(self, reducers: Optional[Dict[str, ReducerFunc]] = None) -> None:
        self._nodes: Dict[str, NodeFunc] = {}
        self._edges: Dict[str, str] = {}
        self._conditional_edges: Dict[str, Tuple[RouterFunc, Dict[str, str]]] = {}
        self._reducers: Dict[str, ReducerFunc] = reducers or {}

    def add_node(self, name: str, func: NodeFunc) -> "StateGraph":
        if name in (START, END):
            raise ValueError(f"禁止使用保留节点名: {name}")
        self._nodes[name] = func
        return self

    def add_edge(self, src: str, dst: str) -> "StateGraph":
        self._edges[src] = dst
        return self

    def add_conditional_edges(
        self,
        src: str,
        router_fn: RouterFunc,
        path_map: Dict[str, str],
    ) -> "StateGraph":
        self._conditional_edges[src] = (router_fn, path_map)
        return self

    def compile(self) -> CompiledGraph:
        """编译期拓扑完整性校验。"""
        valid_targets: Set[str] = set(self._nodes.keys()) | {END}
        valid_sources: Set[str] = set(self._nodes.keys()) | {START}

        if START not in self._edges and START not in self._conditional_edges:
            raise ValueError("状态图必须配置从 START 出发的入口边")

        for src, dst in self._edges.items():
            if src not in valid_sources or dst not in valid_targets:
                raise ValueError(f"非法普通边: {src} -> {dst}")

        for src, (_, path_map) in self._conditional_edges.items():
            if src not in valid_sources:
                raise ValueError(f"非法条件边起点: {src}")
            for dst in path_map.values():
                if dst not in valid_targets:
                    raise ValueError(f"非法条件边终点: {src} -> {dst}")

        return CompiledGraph(
            nodes=dict(self._nodes),
            edges=dict(self._edges),
            conditional_edges=dict(self._conditional_edges),
            reducers=dict(self._reducers),
        )


if __name__ == "__main__":
    g = StateGraph(reducers={"logs": lambda old, new: old + new})
    g.add_node("step_a", lambda s: {"logs": ["A完成"], "val": s["val"] + 1})
    g.add_node("step_b", lambda s: {"logs": ["B完成"], "val": s["val"] * 10})
    g.add_edge(START, "step_a")
    g.add_edge("step_a", "step_b")
    g.add_edge("step_b", END)
    app = g.compile()
    print(app.draw_ascii())
    print("运行结果:", app.invoke({"val": 2, "logs": []}))


### 🔍 步骤 1 代码深度解析

1. `StateGraph.compile()` 在真正运行前就校验了 `START` 入口以及每条边的起终点是否注册，杜绝“上线跑一半才发现节点名拼错”的低级故障。
2. `_merge_state` 实现了与 LangGraph `Annotated[..., operator.add]` 相同的 Reducer 归约逻辑，且使用 `copy.deepcopy` 避免节点函数就地修改污染上游状态。
3. `draw_mermaid()` 用 `-->` 表示普通边、用 `-. label .->` 表示条件边，与官方 LangGraph 可视化风格完全一致。

**预期运行输出参考：**
```text
+--- StateGraph Topology ---+
| [__start__] ===> [step_a]
| [step_a] ===> [step_b]
| [step_b] ===> [__end__]
+---------------------------+
运行结果: {'val': 30, 'logs': ['A完成', 'B完成'], '_visited_nodes': ['step_a', 'step_b']}
```


## 💻 实操步骤 2：步骤二：用状态图重写 ReAct Agent 并实现企业人工审批工作流

- **目标文件**：`versions/day41/src/day41_react_and_approval.py`
- **老陈代码旁白**：引擎造好了，立刻上两道企业硬菜！第一道菜：把 Day 39 的 ReAct 循环用 `StateGraph` 重新编排，体验循环图的丝滑；第二道菜：星辰数科 IT 设备采购工作流——5000 元以内自动过审，超过 5000 元强制进入 `human_approval` 节点由审批回调决定放行或驳回。


In [ ]:
"""Day 41 - 步骤二：基于 StateGraph 的 ReAct 循环图与企业采购审批工作流。"""

from __future__ import annotations

import operator
from typing import Any, Callable, Dict

try:
    from src.day41_state_graph import CompiledGraph, END, START, StateGraph
except ImportError:
    from day41_state_graph import CompiledGraph, END, START, StateGraph


# =========================================================================
# 场景一：用 StateGraph 重写 ReAct Agent (agent <-> tools 循环图)
# =========================================================================

def build_react_graph() -> CompiledGraph:
    """构建标准 LangGraph 风格的 ReAct 状态图。"""
    inventory_db = {"MacBook Pro": 14999, "显示器": 2200, "机械键盘": 650}

    def agent_node(state: Dict[str, Any]) -> Dict[str, Any]:
        question = state["question"]
        tool_results = state.get("tool_results", [])
        # 如果还没查过价格，先发出工具调用指令
        if not tool_results:
            for item_name in inventory_db:
                if item_name in question:
                    return {
                        "pending_tool": {"name": "query_price", "arg": item_name},
                        "messages": [f"Agent思考: 需要查询 {item_name} 的采购单价"],
                    }
            return {
                "pending_tool": None,
                "final_answer": "未识别到库存目录中的设备名称。",
                "messages": ["Agent思考: 无需工具，直接回复"],
            }

        # 已拿到工具结果，汇总得出最终答案
        latest = tool_results[-1]
        return {
            "pending_tool": None,
            "final_answer": f"查询完成：{latest}。如需提交采购申请请告知数量。",
            "messages": ["Agent思考: 已获得价格信息，生成最终答复"],
        }

    def tools_node(state: Dict[str, Any]) -> Dict[str, Any]:
        call = state["pending_tool"]
        item_name = call["arg"]
        price = inventory_db.get(item_name, -1)
        result_str = f"{item_name} 企业协议采购价为 {price} 元"
        return {
            "tool_results": [result_str],
            "messages": [f"Tools执行: query_price({item_name}) -> {result_str}"],
        }

    def should_continue(state: Dict[str, Any]) -> str:
        return "tools" if state.get("pending_tool") else "end"

    graph = StateGraph(reducers={"messages": operator.add, "tool_results": operator.add})
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tools_node)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
    graph.add_edge("tools", "agent")
    return graph.compile()


# =========================================================================
# 场景二：带人工审批节点的企业 IT 采购工作流
# =========================================================================

def build_procurement_approval_graph(
    human_approver: Callable[[Dict[str, Any]], Dict[str, Any]],
    auto_approve_limit: float = 5000.0,
) -> CompiledGraph:
    """
    企业采购审批状态图：
    - 金额 <= auto_approve_limit：自动放行 -> 创建采购单
    - 金额 > auto_approve_limit：强制路由至人工审批节点 -> 通过则下单，驳回则通知
    """

    def parse_request(state: Dict[str, Any]) -> Dict[str, Any]:
        total = float(state["unit_price"]) * int(state["quantity"])
        return {
            "total_amount": total,
            "audit_logs": [f"解析申请: {state['applicant']} 申请采购 {state['item']} x{state['quantity']}，总价 {total:.2f} 元"],
        }

    def route_by_amount(state: Dict[str, Any]) -> str:
        return "need_human" if state["total_amount"] > auto_approve_limit else "auto"

    def auto_approve(state: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "approved": True,
            "approver": "SystemAutoRule",
            "audit_logs": [f"金额未超 {auto_approve_limit} 元红线，系统自动审批通过"],
        }

    def human_approval(state: Dict[str, Any]) -> Dict[str, Any]:
        decision = human_approver(state)
        approved = bool(decision.get("approved", False))
        comment = decision.get("comment", "")
        approver = decision.get("approver", "财务总监")
        return {
            "approved": approved,
            "approver": approver,
            "approval_comment": comment,
            "audit_logs": [f"人工审批({approver}): {'通过' if approved else '驳回'} - {comment}"],
        }

    def route_after_human(state: Dict[str, Any]) -> str:
        return "approved" if state.get("approved") else "rejected"

    def execute_order(state: Dict[str, Any]) -> Dict[str, Any]:
        po_id = f"PO-2026-{abs(hash(state['item'])) % 10000:04d}"
        return {
            "order_status": "ORDERED",
            "po_id": po_id,
            "audit_logs": [f"已调用 ERP 创建采购单 {po_id}"],
        }

    def reject_notify(state: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "order_status": "REJECTED",
            "audit_logs": [f"已向 {state['applicant']} 发送驳回通知：{state.get('approval_comment', '无')}"],
        }

    g = StateGraph(reducers={"audit_logs": operator.add})
    g.add_node("parse_request", parse_request)
    g.add_node("auto_approve", auto_approve)
    g.add_node("human_approval", human_approval)
    g.add_node("execute_order", execute_order)
    g.add_node("reject_notify", reject_notify)

    g.add_edge(START, "parse_request")
    g.add_conditional_edges(
        "parse_request",
        route_by_amount,
        {"auto": "auto_approve", "need_human": "human_approval"},
    )
    g.add_edge("auto_approve", "execute_order")
    g.add_conditional_edges(
        "human_approval",
        route_after_human,
        {"approved": "execute_order", "rejected": "reject_notify"},
    )
    g.add_edge("execute_order", END)
    g.add_edge("reject_notify", END)
    return g.compile()


if __name__ == "__main__":
    react_app = build_react_graph()
    print("=== ReAct Mermaid 拓扑 ===")
    print(react_app.draw_mermaid())
    r = react_app.invoke({"question": "查一下 MacBook Pro 采购价", "messages": [], "tool_results": []})
    print("ReAct 结果:", r["final_answer"], "| 经过节点:", r["_visited_nodes"])

    approval_app = build_procurement_approval_graph(
        human_approver=lambda s: {"approved": True, "approver": "王总监", "comment": "项目急需，同意采购"}
    )
    p = approval_app.invoke(
        {"applicant": "阿杰", "item": "MacBook Pro", "unit_price": 14999, "quantity": 1, "audit_logs": []}
    )
    print("采购审批状态:", p["order_status"], "| 经过节点:", p["_visited_nodes"])


### 🔍 步骤 2 代码深度解析

1. 在 `build_react_graph` 中，`agent -> tools -> agent` 构成了显式闭环，且 `messages` 与 `tool_results` 通过 `operator.add` Reducer 自动累积，代码比手写 `while` 循环清爽了一个量级。
2. 在 `build_procurement_approval_graph` 中，大额审批由 `route_by_amount` 这段确定性 Python 代码把关，完全杜绝大模型“嘴瓢直接放行”的合规漏洞。
3. `human_approver` 以依赖注入的方式传入，方便单测模拟“同意”和“驳回”两种审批结果；明天我们会进一步把它升级为真正的异步挂起/恢复（Checkpointer + Interrupt）。

**预期运行输出参考：**
```text
=== ReAct Mermaid 拓扑 ===
graph TD
    __start__ --> agent
    tools --> agent
    agent -. tools .-> tools
    agent -. end .-> __end__
ReAct 结果: 查询完成：MacBook Pro 企业协议采购价为 14999 元。如需提交采购申请请告知数量。 | 经过节点: ['agent', 'tools', 'agent']
采购审批状态: ORDERED | 经过节点: ['parse_request', 'human_approval', 'execute_order']
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day41/tests/test_day41.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 41 自动化单元测试：验证 StateGraph 引擎、可视化导出、ReAct 状态图与采购审批分支。"""

import pytest
from src.day41_state_graph import StateGraph, START, END
from src.day41_react_and_approval import build_react_graph, build_procurement_approval_graph


def test_state_graph_compile_validation_and_visualization():
    g = StateGraph()
    g.add_node("a", lambda s: {"x": 1})
    g.add_edge(START, "a")
    g.add_edge("a", END)
    app = g.compile()
    assert "graph TD" in app.draw_mermaid()
    assert "__start__ --> a" in app.draw_mermaid()
    assert "[__start__] ===> [a]" in app.draw_ascii()

    bad = StateGraph()
    bad.add_edge(START, "missing_node")
    with pytest.raises(ValueError):
        bad.compile()


def test_react_state_graph_loop():
    app = build_react_graph()
    res = app.invoke({"question": "帮我查显示器单价", "messages": [], "tool_results": []})
    assert "2200" in res["final_answer"]
    assert res["_visited_nodes"] == ["agent", "tools", "agent"]


def test_procurement_auto_approve_branch():
    called = {"human": False}

    def approver(_s):
        called["human"] = True
        return {"approved": True}

    app = build_procurement_approval_graph(human_approver=approver, auto_approve_limit=5000)
    res = app.invoke({"applicant": "小李", "item": "机械键盘", "unit_price": 650, "quantity": 2, "audit_logs": []})
    assert res["order_status"] == "ORDERED"
    assert called["human"] is False
    assert res["_visited_nodes"] == ["parse_request", "auto_approve", "execute_order"]


def test_procurement_human_approve_and_reject_branches():
    app_ok = build_procurement_approval_graph(
        human_approver=lambda _s: {"approved": True, "approver": "总监", "comment": "同意"}
    )
    res_ok = app_ok.invoke({"applicant": "阿杰", "item": "MacBook Pro", "unit_price": 14999, "quantity": 1, "audit_logs": []})
    assert res_ok["order_status"] == "ORDERED"
    assert "human_approval" in res_ok["_visited_nodes"]

    app_rej = build_procurement_approval_graph(
        human_approver=lambda _s: {"approved": False, "approver": "总监", "comment": "预算不足"}
    )
    res_rej = app_rej.invoke({"applicant": "阿杰", "item": "MacBook Pro", "unit_price": 14999, "quantity": 2, "audit_logs": []})
    assert res_rej["order_status"] == "REJECTED"
    assert res_rej["_visited_nodes"] == ["parse_request", "human_approval", "reject_notify"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为采购工作流增加二级审批（超过 2 万元需 VP 复审）

**题目要求：**
扩展 `build_procurement_approval_graph`：金额 <= 5000 自动通过；5000 < 金额 <= 20000 走总监审批；金额 > 20000 在总监通过后还必须进入 `vp_approval` 节点复审。

**思路提示：** 在 `human_approval` 后的条件边中判断 `state['approved']` 与 `state['total_amount'] > 20000`，增加 `vp_approval` 节点。


In [ ]:
"""Day 41 作业 1 参考答案：二级分级审批路由逻辑。"""

def route_after_director(state: dict) -> str:
    if not state.get("approved"):
        return "rejected"
    if state["total_amount"] > 20000:
        return "need_vp"
    return "approved"


if __name__ == "__main__":
    assert route_after_director({"approved": False, "total_amount": 30000}) == "rejected"
    assert route_after_director({"approved": True, "total_amount": 12000}) == "approved"
    assert route_after_director({"approved": True, "total_amount": 25000}) == "need_vp"
    print("二级审批条件边测试通过")


**答案解析与评分标准：**

利用条件边清晰表达多级审批矩阵，比嵌套 `if/else` 更易扩展，且能自动反映在 Mermaid 拓扑图中。


### 作业 2（企业实战）：实现自定义字典合并 Reducer（支持统计各节点耗时）

**题目要求：**
为 `StateGraph` 编写一个 `merge_metrics_reducer(old: dict, new: dict) -> dict`，使每个节点返回的 `{'node_metrics': {'node_name': ms}}` 能合并保留所有历史节点指标而非互相覆盖。

**思路提示：** 返回 `{**old, **new}` 浅拷贝合并字典，避免原地修改 `old`。


In [ ]:
"""Day 41 作业 2 参考答案：指标字典合并 Reducer。"""

from typing import Dict


def merge_metrics_reducer(old: Dict[str, float], new: Dict[str, float]) -> Dict[str, float]:
    merged = dict(old)
    merged.update(new)
    return merged


if __name__ == "__main__":
    s1 = merge_metrics_reducer({"parse": 1.2}, {"approve": 5.4})
    s2 = merge_metrics_reducer(s1, {"order": 2.1})
    print("合并指标:", s2)
    assert s2 == {"parse": 1.2, "approve": 5.4, "order": 2.1}


**答案解析与评分标准：**

Reducer 不仅限于 `operator.add` 拼接列表，针对 `dict` 或 `set` 自定义无副作用合并函数，是 LangGraph 复杂状态管理的常用技巧。


### 作业 3（架构进阶）：为 StateGraph 增加图可达性静态检测（发现孤岛节点）

**题目要求：**
编写函数 `find_unreachable_nodes(nodes, edges, conditional_edges)`，使用 BFS 从 `START` 遍历，找出所有已注册但永远无法被执行到的“孤岛节点”，在 `compile()` 时发出警告。

**思路提示：** 构建邻接表 `adj[src] -> list[dst]`，从 `START` 做标准队列 BFS，用 `set(nodes) - visited` 得到不可达节点集合。


In [ ]:
"""Day 41 作业 3 参考答案：BFS 静态检测状态图孤岛节点。"""

from collections import deque
from typing import Dict, Set


def find_unreachable_nodes(
    node_names: Set[str],
    edges: Dict[str, str],
    cond_edges: Dict[str, Dict[str, str]],
) -> Set[str]:
    adj: Dict[str, Set[str]] = {}
    for src, dst in edges.items():
        adj.setdefault(src, set()).add(dst)
    for src, pmap in cond_edges.items():
        adj.setdefault(src, set()).update(pmap.values())

    visited: Set[str] = set()
    q = deque(["__start__"])
    while q:
        cur = q.popleft()
        for nxt in adj.get(cur, set()):
            if nxt not in visited:
                visited.add(nxt)
                q.append(nxt)
    return node_names - visited


if __name__ == "__main__":
    orphans = find_unreachable_nodes(
        node_names={"a", "b", "dead_node"},
        edges={"__start__": "a", "a": "b", "b": "__end__"},
        cond_edges={},
    )
    print("发现孤岛节点:", orphans)
    assert orphans == {"dead_node"}


**答案解析与评分标准：**

大型多 Agent 图常有十几个节点，重构时极易残留未被连线的死节点。引入 BFS 可达性检测能把拓扑缺陷消灭在 CI 阶段。


---

## 🎯 老陈收尾复盘

今天这一仗非常关键！把 `StateGraph` 吃透之后，你再回头看真实 LangGraph 的 `add_node`、`add_conditional_edges`、`compile()`、`draw_mermaid()`，会发现每一个 API 都亲切得像老朋友。回去把审批工作流的边界条件再测两遍，明天我们给它装上“断点续跑”和“多线程并行加速”引擎！
